# CBIS-DDSM preprocessing

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
os.chdir(REPO_ROOT)  
sys.path.insert(0, str(REPO_ROOT))

## Load raw data

In [ ]:
from preprocessing.cbis_ddsm import CbisDDSM, csv_save_path
from utils.preprocessing import birads_assessment, birads_mapping, breast_density, dview, get_value, laterality

ds = CbisDDSM()
ds.set_dataset_name("cbis-ddsm")
ds.full_df.shape, ds.full_df.head()

### Step 1 — drop duplicate abnormality rows

In [ ]:
ds.full_df.drop_duplicates(
    subset=["patient id", "breast density", "left or right breast", "image view", "abnormality type",
            "calc type", "calc distribution", "assessment", "pathology", "image file path",
            "mass shape", "mass margins"],
    inplace=True,
)
ds.full_df.shape

### Step 2/3 — null out fields that don't apply to the row's abnormality type

In [ ]:
ds.full_df.loc[ds.full_df["abnormality type"] == "calcification", ["mass shape", "mass margins"]] = None
ds.full_df.loc[ds.full_df["abnormality type"] == "mass", ["calc type", "calc distribution"]] = None
ds.full_df[["mass shape", "calc type"]].isna().sum()

### Step 4 — rename columns for consistency

In [ ]:
ds.full_df = ds.full_df.rename(columns={
    "left or right breast": "laterality",
    "calc type": "calcification type",
    "calc distribution": "calcification distribution",
})
ds.full_df.columns.tolist()

### Step 5 — map laterality, view, density, and assessment to harmonized labels

In [ ]:
ds.full_df["laterality"] = ds.full_df["laterality"].apply(lambda x: get_value(x, laterality))
ds.full_df["image view"] = ds.full_df["image view"].apply(lambda x: get_value(x, dview))
ds.full_df["breast density"] = ds.full_df["breast density"].apply(lambda x: get_value(x, breast_density))
ds.full_df["assessment"] = ds.full_df["assessment"].apply(lambda x: get_value(x, birads_assessment))
ds.full_df["assessment"].value_counts(dropna=False)

### Step 6 — drop additional-evaluation (BI-RADS 0) rows

In [ ]:
ds.full_df = ds.full_df[ds.full_df["assessment"] != birads_mapping[0]]
ds.full_df.shape

### Step 7 — second rename pass for the common schema

In [ ]:
ds.full_df = ds.full_df.rename(columns={"image view": "exam view", "mass margins": "mass margin"})
ds.full_df.columns.tolist()

## Build one exam record

In [ ]:
row = ds.full_df.iloc[0]
exam = ds.process_row(row)
exam

## Final processed CSV

In [ ]:
import json

final_df = pd.read_csv(csv_save_path)
print(f"rows before per-exam processing: {len(ds.full_df)} | rows in final csv: {len(final_df)}")
final_df.head()

In [ ]:
sample = final_df.iloc[0]
print(json.dumps(json.loads(sample['context']), indent=2))
print(json.dumps(json.loads(sample['findings']), indent=2))